# Canonical Provider Boundary (Slice 5A)

Through Feature 4, every OwnerLens calculation read raw SEC Company Facts directly and resolved SEC XBRL concepts itself. Slice 5A inserts one boundary:

```
providers (SEC today; FMP in Slice 5B)
   ↓  provider adapter
CanonicalFinancialHistory  (provider-neutral facts + provenance + per-metric status)
   ↓
OwnerLens deterministic logic  (owner economics, capital efficiency, Feature 2, coverage)
```

This slice changes **where facts come from**, not what the economics mean: FCF, ROIC, margins, growth, classifications, and coverage are unchanged. This notebook walks one company from raw SEC facts → canonical facts → owner economics, then shows the same economics computed from a synthetic non-SEC history.

In [1]:
import json
import os
from datetime import date

from owner_lens import (
    CANONICAL_METRICS,
    CanonicalFact,
    CanonicalFinancialHistory,
    SecClient,
    canonical_history_from_sec,
    compute_content_hash,
    owner_economics_from_history,
)
from owner_lens.canonical import MetricKind, metric_spec

USER_AGENT = os.environ.get('OWNER_LENS_SEC_USER_AGENT', 'OwnerLens admin@example.com')

with SecClient(USER_AGENT) as client:
    results = {t: client.retrieve_company_facts(t) for t in ('ADBE', 'V')}

# Retrieval context: enough to reconcile against the exact payload later.
for ticker, result in results.items():
    payload = json.dumps(result.raw_facts, sort_keys=True).encode()
    print(f'{ticker}: CIK {result.identity.cik}  content hash {compute_content_hash(payload)}')

ADBE: CIK 0000796343  content hash 4241d1761736b282aef623c173b6a34f817906d24ef87106c64aa4da4bff8e23
V: CIK 0001403161  content hash 783a56fbf22230098570c580e6452182652893ea64d764b2ba0e3c1c91085042


## 1. Raw SEC facts

Below SEC sits the raw XBRL shape: concept name → unit → a list of observations, including quarterly values, comparative repeats, and restatements. OwnerLens selects one concept per metric and resolves full fiscal years deterministically. This is SEC-specific knowledge, and after Slice 5A it lives **only above** the canonical boundary.

In [2]:
adbe_raw = results['ADBE'].raw_facts
adbe = canonical_history_from_sec(adbe_raw, ticker='ADBE')
revenue_concept = adbe.series_for('revenue').observations[0].provider_field

raw_fy = [
    f for f in adbe_raw['facts']['us-gaap'][revenue_concept]['units']['USD']
    if f.get('fp') == 'FY' and f.get('form', '').startswith('10-K')
]
print(f'{revenue_concept}: {len(raw_fy)} raw FY observations (including comparative repeats)')
for f in sorted(raw_fy, key=lambda f: (f['end'], f['filed']))[-6:]:
    print(f"  {f['start']} → {f['end']}  val={f['val']:>15,}  filed={f['filed']}  accn={f['accn']}")

Revenues: 139 raw FY observations (including comparative repeats)
  2022-12-03 → 2023-12-01  val= 19,409,000,000  filed=2024-01-17  accn=0000796343-24-000006
  2022-12-03 → 2023-12-01  val= 19,409,000,000  filed=2025-01-13  accn=0000796343-25-000004
  2022-12-03 → 2023-12-01  val= 19,409,000,000  filed=2026-01-15  accn=0000796343-26-000003
  2023-12-02 → 2024-11-29  val= 21,505,000,000  filed=2025-01-13  accn=0000796343-25-000004
  2023-12-02 → 2024-11-29  val= 21,505,000,000  filed=2026-01-15  accn=0000796343-26-000003
  2024-11-30 → 2025-11-28  val= 23,769,000,000  filed=2026-01-15  accn=0000796343-26-000003


## 2. Canonical facts with provenance

`canonical_history_from_sec` reuses the existing SEC normalizers unchanged and emits one `CanonicalFact` per metric and fiscal year. The metric name is OwnerLens vocabulary (`revenue`), never an XBRL tag; the SEC concept survives only as **provenance** in `provider_field`, alongside the provider, form, filing date, and accession.

In [3]:
print(f"{'metric':<8} {'FY':>4} {'value':>16}  {'provider':<8} {'provider_field':<14} {'form':<6} {'filed':<10}  accession")
for fact in adbe.series_for('revenue').observations:
    print(
        f'{fact.metric:<8} {fact.fiscal_year:>4} {fact.value:>16,}  {fact.provider:<8} '
        f'{fact.provider_field:<14} {fact.form:<6} {fact.filed!s:<10}  {fact.accession}'
    )

cash = adbe.series_for('cash').observations[0]
print()
print('instant fact period_start:', cash.period_start, '| period_end:', cash.period_end)

metric     FY            value  provider provider_field form   filed       accession
revenue  2025   23,769,000,000  sec      Revenues       10-K   2026-01-15  0000796343-26-000003
revenue  2024   21,505,000,000  sec      Revenues       10-K   2025-01-13  0000796343-25-000004
revenue  2023   19,409,000,000  sec      Revenues       10-K   2024-01-17  0000796343-24-000006
revenue  2022   17,606,000,000  sec      Revenues       10-K   2023-01-17  0000796343-23-000007
revenue  2021   15,785,000,000  sec      Revenues       10-K   2022-01-21  0000796343-22-000032

instant fact period_start: None | period_end: 2025-11-28


## 3. Every metric carries an explicit status

Each of the 17 canonical metrics is `AVAILABLE`, `STRUCTURALLY_ABSENT` (for example, no dividend program), `UNSUPPORTED` (no trusted provider field), or `INVALID` (malformed or ambiguous provider data). Failures are stored, not raised: a calculation fails only when it **requires** an unusable metric, with the provider's original typed error. Visa's missing diluted-share concept is the canonical example.

In [4]:
visa = canonical_history_from_sec(results['V'].raw_facts, ticker='V')

print(f"{'metric':<26} {'ADBE':<20} {'V':<20}")
for spec in CANONICAL_METRICS:
    print(f'{spec.name:<26} {adbe.statuses()[spec.name].value:<20} {visa.statuses()[spec.name].value:<20}')

print()
print('V diluted_shares reason:', visa.series_for('diluted_shares').reason)

metric                     ADBE                 V                   
revenue                    AVAILABLE            AVAILABLE           
operating_income           AVAILABLE            AVAILABLE           
net_income                 AVAILABLE            AVAILABLE           
operating_cash_flow        AVAILABLE            AVAILABLE           
capital_expenditures       AVAILABLE            AVAILABLE           
diluted_shares             AVAILABLE            UNSUPPORTED         
income_tax_expense         AVAILABLE            AVAILABLE           
pretax_income              AVAILABLE            AVAILABLE           
repurchases                AVAILABLE            AVAILABLE           
stock_based_compensation   AVAILABLE            AVAILABLE           
dividends_paid             STRUCTURALLY_ABSENT  AVAILABLE           
cash                       AVAILABLE            AVAILABLE           
short_term_investments     AVAILABLE            STRUCTURALLY_ABSENT 
current_debt               AVAILAB

## 4. OwnerLens economics from canonical history

`owner_economics_from_history` accepts only a `CanonicalFinancialHistory`. It never sees SEC JSON or XBRL tags, yet every row still links back to its source filing through the canonical facts it carries.

In [5]:
rows = owner_economics_from_history(adbe)
print(f"{'FY':>4} {'revenue':>16} {'FCF':>16} {'FCF/share':>10} {'FCF margin':>11}  revenue source")
for row in rows:
    fps = f'{row.fcf_per_share:.2f}' if row.fcf_per_share is not None else 'n/a'
    margin = f'{row.fcf_margin:.1%}' if row.fcf_margin is not None else 'n/a'
    src = f'{row.revenue.provider}:{row.revenue.provider_field} ({row.revenue.accession})' if row.revenue else 'n/a'
    print(f'{row.fiscal_year:>4} {row.revenue.value if row.revenue else 0:>16,} {row.free_cash_flow or 0:>16,} {fps:>10} {margin:>11}  {src}')

  FY          revenue              FCF  FCF/share  FCF margin  revenue source
2025   23,769,000,000    9,852,000,000      23.07       41.4%  sec:Revenues (0000796343-26-000003)
2024   21,505,000,000    7,873,000,000      17.51       36.6%  sec:Revenues (0000796343-25-000004)
2023   19,409,000,000    6,942,000,000      15.12       35.8%  sec:Revenues (0000796343-24-000006)
2022   17,606,000,000    7,396,000,000      15.71       42.0%  sec:Revenues (0000796343-23-000007)
2021   15,785,000,000    6,882,000,000      14.31       43.6%  sec:Revenues (0000796343-22-000032)


## 5. Same economics, no SEC at all

To prove the boundary, build a tiny history by hand with `provider="example"` and no SEC payload. The same deterministic functions compute owner economics. This is exactly the seam Slice 5B uses to plug in a second provider.

In [6]:
def fact(metric: str, year: int, value: int) -> CanonicalFact:
    duration = metric_spec(metric).kind is MetricKind.DURATION
    return CanonicalFact(
        metric=metric, value=value, unit=metric_spec(metric).unit,
        fiscal_year=year, fiscal_period='FY',
        period_end=date(year, 12, 31), period_start=date(year, 1, 1) if duration else None,
        provider='example', provider_field=f'example.{metric}',
        form='annual-report', filed=date(year + 1, 2, 15), accession=f'ex-{metric}-{year}',
    )

values = {
    'revenue': {2024: 1000, 2025: 1200}, 'operating_income': {2024: 200, 2025: 300},
    'net_income': {2024: 150, 2025: 240}, 'operating_cash_flow': {2024: 300, 2025: 400},
    'capital_expenditures': {2024: 50, 2025: 100}, 'diluted_shares': {2024: 100, 2025: 100},
}
synthetic = CanonicalFinancialHistory.build(
    ticker='EXAMPLE', max_years=2,
    facts=[fact(m, y, v) for m, by_year in values.items() for y, v in by_year.items()],
    structurally_absent=[s.name for s in CANONICAL_METRICS if s.name not in values],
)
for row in owner_economics_from_history(synthetic):
    print(row.fiscal_year, 'FCF', row.free_cash_flow, 'FCF/share', row.fcf_per_share,
          'FCF growth', row.fcf_growth, '| provider:', row.revenue.provider)
print('providers in history:', sorted(synthetic.providers()))

2025 FCF 300 FCF/share 3.0 FCF growth 0.2 | provider: example
2024 FCF 250 FCF/share 2.5 FCF growth None | provider: example
providers in history: ['example']


## 6. The boundary rule

> Downstream OwnerLens financial logic must not depend on SEC XBRL concept names or raw SEC JSON.

* SEC concept preferences, per-company overrides, period selection, and ambiguity detection live in the SEC normalizers and `sec_adapter.py`.
* The seven analytical modules import only the canonical model; `tests/test_canonical_boundary.py` enforces this with an AST check.
* The legacy `*_from_facts` functions remain as thin compatibility wrappers (SEC → canonical → `*_from_history`) and are slated for removal once all callers migrate.
* A characterization baseline proves ADBE/V/COST outputs, failures, and persisted records are byte-identical to the pre-canonical code.

**Slice 5B outlook:** an FMP adapter produces the same `CanonicalFinancialHistory` with `provider="fmp"` and its own `provider_field` values; no downstream calculation changes.